In [25]:
import pandas as pd
import re

def clean_mistral_lay(text):
    """
    Clean Mistral lay translations:
    - Remove [INST]..[/INST] prompts
    - Remove "Sentence:" parts
    - Keep first translation before "Or,"
    - Strip numbered lists
    """
    if pd.isna(text):
        return ""
    
    text = str(text)
    
    # Remove ENTIRE prompt block: [INST]...[/INST]
    text = re.sub(r'\[INST\].*?\[/INST\]\s*', '', text, flags=re.DOTALL | re.IGNORECASE)
    
    # Remove "Sentence:" part if present
    text = re.sub(r'Sentence:\s*.*?\n?', '', text, flags=re.DOTALL | re.IGNORECASE)
    
    # Keep FIRST translation only (before first "Or,")
    if 'Or,' in text:
        main = text.split('Or,', 1)[0].strip()
    else:
        main = text.strip()
    
    # Clean numbered lists (1., 2., etc.)
    main = re.sub(r'\s*\d+\.\s*', '', main)
    
    return main.strip()

# Load data
df = pd.read_csv('/Users/joannehui/Desktop/fyp/padchest/Reports_public/reports_500_translated_mistral_lay.csv')

# Apply cleaning
df['mistral_clean_lay'] = df['lay_translation'].apply(clean_mistral_lay)

# Stats
print("✅ Cleaning complete:")
print(f"Original:  {df['lay_translation'].str.len().mean():.0f} ± {df['lay_translation'].str.len().std():.0f} chars")
print(f"Clean:     {df['mistral_clean_lay'].str.len().mean():.0f} ± {df['mistral_clean_lay'].str.len().std():.0f} chars")

# Sample
print("\nSample cleaning:")
print(df[['sentence_en', 'lay_translation', 'mistral_clean_lay']].head(3))

# Save
df[['StudyID', 'ImageID', 'sentence_en', 'mistral_clean_lay']].to_csv('mistral_clean_lay.csv', index=False)
print("\n✅ Saved: mistral_clean_lay.csv")


✅ Cleaning complete:
Original:  537 ± 77 chars
Clean:     180 ± 113 chars

Sample cleaning:
                                         sentence_en  \
0               Minimal biapical pleural thickening.   
1  Slight blunting of the posterior left costophr...   
2                                                NaN   

                                     lay_translation  \
0  [INST] You are a helpful medical translator. T...   
1  [INST] You are a helpful medical translator. T...   
2                                                NaN   

                                   mistral_clean_lay  
0  "Your X-ray shows a slight thickening of the l...  
1  The back left corner of your ribcage is slight...  
2                                                     

✅ Saved: mistral_clean_lay.csv
